In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from math import ceil
from scipy import stats

In [29]:
filename = ("US-Ne1", "US-Ne2", "US-Ne3", "US-Var")

compare_vars = ["geeSEBAL", "PT-JPL", "SSEBop", "SIMS", "eeMETRIC", "DisALEXI", "Ensemble", "SIF_ET", "Climatology"]


for fn in filename:
    df = pd.read_csv(fn+"_clim.csv", delimiter = ',', parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"])
    df.set_index("Date", inplace=True)
    season_filters = {
        "All Months": df,
        "DJF": df[df.index.month.isin([12, 1, 2])],
        "MAM": df[df.index.month.isin([3, 4, 5])],
        "JJA": df[df.index.month.isin([6, 7, 8])],
        "SON": df[df.index.month.isin([9, 10, 11])],
        }
    season_names = list(season_filters.keys())
    results = []


    for row_idx, season_name in enumerate(season_names):
        season_df = season_filters[season_name]

        for col_idx, var in enumerate(compare_vars):
            subset = season_df[["observed_ET", var]].dropna()
            if subset.empty:
                continue

            y = subset[var].values.reshape(-1, 1)
            obs = subset["observed_ET"].values

            model = LinearRegression().fit(y, obs)
            y_pred = model.predict(y)
            r2 = r2_score(obs, y_pred)
            
            y = y.reshape(-1, )
            y_pred = y_pred.reshape(-1, )
            
            RMSE = np.sqrt(mean_squared_error(obs, y))
            MAE = mean_absolute_error(obs, y)
            MBE = (sum(y-obs))/len(obs)
            SRMSE = RMSE/(np.std(obs))

            results.append({
                "Season": season_name,
                "Variable": var,
                "R2": round(r2, 3),
                "RMSE": round(RMSE, 3),
                "MAE": round(MAE,3),
                "MBE": round(MBE,3),
                "# Months": len(subset),
                "# Years": len(subset.index.to_period("Y").unique())
            })
            season_r2_dict[var] = round(r2, 3)
            season_RMSE_dict[var] = round(RMSE, 3)

    results_df = pd.DataFrame(results)
    display(results_df)
    results_df.to_csv(fn+'_stats.csv')

,Season,Variable,R2,RMSE,MAE,MBE,# Months,# Years
0,All Months,geeSEBAL,0.869,25.734,19.131,-12.123,203,19
1,All Months,PT-JPL,0.893,23.851,16.917,-9.879,203,19
2,All Months,SSEBop,0.880,24.971,19.821,-9.988,203,19
3,All Months,SIMS,0.875,22.242,17.152,2.106,203,19
4,All Months,eeMETRIC,0.837,27.071,21.198,-5.964,203,19
5,All Months,DisALEXI,0.916,19.942,14.685,-7.184,203,19
6,All Months,Ensemble,0.923,19.546,15.142,-8.868,203,19
7,All Months,SIF_ET,0.878,31.130,21.373,-11.272,223,19
8,All Months,Climatology,0.944,14.658,10.314,0.000,223,19
9,DJF,geeSEBAL,0.105,11.957,9.821,-7.837,48,19


,Season,Variable,R2,RMSE,MAE,MBE,# Months,# Years
0,All Months,geeSEBAL,0.874,25.580,19.091,-14.048,202,19
1,All Months,PT-JPL,0.895,21.801,14.237,-7.438,202,19
2,All Months,SSEBop,0.875,25.167,20.488,-10.519,202,19
3,All Months,SIMS,0.861,23.197,17.878,5.555,202,19
4,All Months,eeMETRIC,0.831,27.853,21.763,-8.261,202,19
5,All Months,DisALEXI,0.924,19.897,15.208,-10.020,202,19
6,All Months,Ensemble,0.922,19.279,15.061,-9.517,202,19
7,All Months,SIF_ET,0.869,27.597,18.643,-5.500,223,19
8,All Months,Climatology,0.929,15.842,9.966,0.000,223,19
9,DJF,geeSEBAL,0.061,11.406,8.983,-7.136,46,18


,Season,Variable,R2,RMSE,MAE,MBE,# Months,# Years
0,All Months,geeSEBAL,0.855,22.115,16.636,-8.946,197,18
1,All Months,PT-JPL,0.887,17.930,12.642,-0.469,197,18
2,All Months,SSEBop,0.886,21.212,16.839,-6.783,197,18
3,All Months,SIMS,0.854,25.145,18.889,12.851,197,18
4,All Months,eeMETRIC,0.837,24.103,18.242,-2.913,197,18
5,All Months,DisALEXI,0.892,19.137,14.764,-4.016,197,18
6,All Months,Ensemble,0.917,16.015,12.222,-3.680,197,18
7,All Months,SIF_ET,0.871,20.396,15.779,2.881,222,19
8,All Months,Climatology,0.931,13.609,8.707,0.000,222,19
9,DJF,geeSEBAL,0.119,9.180,7.801,-5.362,46,17


,Season,Variable,R2,RMSE,MAE,MBE,# Months,# Years
0,All Months,geeSEBAL,0.208,43.858,30.882,25.096,200,18
1,All Months,PT-JPL,0.696,32.698,27.477,27.326,200,18
2,All Months,SSEBop,0.412,27.039,18.985,10.624,200,18
3,All Months,eeMETRIC,0.743,20.882,16.052,12.230,200,18
4,All Months,DisALEXI,0.378,28.662,19.891,11.443,200,18
5,All Months,Ensemble,0.522,26.855,19.317,14.817,200,18
6,All Months,SIF_ET,0.817,28.654,25.151,23.516,246,21
7,All Months,Climatology,0.908,8.871,5.123,0.000,246,21
8,DJF,geeSEBAL,0.321,8.204,6.366,-3.038,50,18
9,DJF,PT-JPL,0.574,15.691,14.507,14.484,50,18


In [30]:
site = ("US-Ne1", "US-Ne2", "US-Ne3", "US-Var")
plot_vars = ["ens_anom", "SIF_anom"]
results = []
for fn in site:
    df = pd.read_csv(fn+"_clim.csv", delimiter = ',', parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"])
    df.set_index("Date", inplace=True)
    season_filters = {
        "All Months": df,
        "DJF": df[df.index.month.isin([12, 1, 2])],
        "MAM": df[df.index.month.isin([3, 4, 5])],
        "JJA": df[df.index.month.isin([6, 7, 8])],
        "SON": df[df.index.month.isin([9, 10, 11])],
        }
    season_names = list(season_filters.keys())
    for season_name in season_names:
        season_df = season_filters[season_name]
        for var in plot_vars:
            subset = season_df[["obs_anom", var]].dropna()
        
            y = subset[var].values.reshape(-1, 1)
            obs = subset["obs_anom"].values
            
            model = LinearRegression().fit(y, obs)
            y_pred = model.predict(y)
            r2 = r2_score(obs, y_pred)

            y = y.reshape(-1, )

            RMSE = np.sqrt(mean_squared_error(obs, y))
            MAE = mean_absolute_error(obs, y)
            MBE = (sum(y-obs))/len(obs)
            SRMSE = RMSE/(np.std(obs))
            
            results.append({
                "Site": fn,
                "Season": season_name,
                "Variable": var,
                "R2": round(r2, 3),
                "RMSE": round(RMSE, 3),
                "SRMSE": round(SRMSE, 3),
                "MAE": round(MAE,3),
                "MBE": '{:0.3e}'.format(MBE),
                "# Months": len(subset),
                "# Years": len(subset.index.to_period("Y").unique()),
            })
            
results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv('anom_stats.csv')

,Site,Season,Variable,R2,RMSE,SRMSE,MAE,MBE,# Months,# Years
0,US-Ne1,All Months,ens_anom,0.304,14.420,0.977,9.862,5.230e-01,203,19
1,US-Ne1,All Months,SIF_anom,0.003,17.081,1.165,11.868,4.040e-02,223,19
2,US-Ne1,DJF,ens_anom,0.167,5.839,0.931,4.605,3.717e-01,48,19
3,US-Ne1,DJF,SIF_anom,0.231,5.833,0.936,4.708,2.435e-02,55,19
4,US-Ne1,MAM,ens_anom,0.304,13.171,1.027,10.583,1.345e+00,49,17
5,US-Ne1,MAM,SIF_anom,0.002,14.468,1.064,11.912,1.414e-01,54,18
6,US-Ne1,JJA,ens_anom,0.367,20.692,0.923,14.200,-2.517e-01,52,19
7,US-Ne1,JJA,SIF_anom,0.045,27.676,1.279,21.622,1.379e-04,57,19
8,US-Ne1,SON,ens_anom,0.161,13.478,1.107,9.703,6.574e-01,54,19
9,US-Ne1,SON,SIF_anom,0.143,12.013,0.951,8.982,5.017e-04,57,19
